In [1]:
import html
from itertools import batched
from pathlib import Path
import pickle
import random
import re

from keybert import KeyBERT
from sentence_transformers import SentenceTransformer
from stop_words import get_stop_words

In [2]:
# Set a fixed seed for reproducible document selection
random = random.Random(42)

First, we compile a list of directories in `data/Cross-Language-Dataset/dataset/sentences` which possess both English and Spanish-language subdirectories. We will skip the Wikipedia corpus since its sentences do not strictly correspond to one another across languages; we are solely interested in directly parallel sentence pairings.

In [3]:
ferrero_sentences = Path('data/Cross-Language-Dataset/dataset/sentences')
es_en_collections = []

# Iterate through dataset/sentences
for collection_dir in ferrero_sentences.iterdir():
    # If collection has a 'Wiki-' like name, skip it
    if (str(collection_dir).lower().find('wiki') > -1):
        continue
    
    # Skip hidden files
    if '/.' in str(collection_dir): 
        continue

    english_found = False
    spanish_found = False

    # Iterate through dataset/sentences/collection_dir
    for language_dir in collection_dir.iterdir():
        # Skip hidden files
        if '/.' in str(collection_dir): 
            continue

        if str(language_dir).endswith('en'):
            english_found = True
        if str(language_dir).endswith('es'):
            spanish_found = True

    # If dataset/sentences/collection_dir contains both English and Spanish data:
    if english_found and spanish_found:
        es_en_collections.append(collection_dir)

Next, we confirm that each collection has an English file analagous to every one of the collection's Spanish files, and vice versa.

In [4]:
# Compile a list of the English- and Spanish-language filepaths in the es_en_collections
en_paths = set()
es_paths = set()

for collection in es_en_collections:
    # Add each es_dir's .txt file to es_paths
    for es_dir in collection.glob('es'):
        for txt_file in es_dir.glob('*.txt'):
            es_paths.add(str(txt_file))

    # Add each en_dir's .txt file to en_paths
    for en_dir in collection.glob('en'):
        for txt_file in en_dir.glob('*.txt'):
            en_paths.add(str(txt_file))



# Check that an English filepath exists for every Spanish filepath, and vice versa
files_ok = True
num_en_files = len(en_paths)
num_es_files = len(es_paths)

if num_en_files != num_es_files:
    print('Warning: unequal proportion of English files to Spanish files')
    files_ok = False

for path in en_paths:
    assumed_es_path = str(path.replace('/en', '/es')).replace('-en', '-es')
    if not assumed_es_path in es_paths:
        print('No Spanish analog found for English file', path)
        files_ok = False

for path in es_paths:
    assumed_en_path = str(path.replace('/es', '/en')).replace('-es', '-en')
    if not assumed_en_path in en_paths:
        print('No English analog found for Spanish file', path)
        files_ok = False

if files_ok:
    print('All English and Spanish files are 1-to-1.')

All English and Spanish files are 1-to-1.


Now that we know each sentence collection has an English file corresponding to every Spanish file (and vice versa), we can safely assume that we can compute an equivalent `es_path` for each `en_path`. We will limit the dataset's sources to a random sample of 100 files with 100+ sentences each and truncate them to exactly 100 sentences. This will give us 10,000 candidate sentences to test against, which should be plenty.

In [5]:
# Randomize the candidate file paths 
en_paths = list(sorted(en_paths))
random.shuffle(en_paths)

We will map each corresponding English and Spanish file into sentence-pair objects that each contain an English sentence, the corresponding Spanish sentence, and key phrases from each. Our dataset will in turn consist of a dictionary which uses each file name as a key and a list of the aforementioned sentence-pair objects as the value. To extract the key phrases in each sentence, we will use KeyBERT and multilingual sentence transformer `paraphrase-multilingual-MiniLM-L12-v2`.

In [6]:
# Set up the key phrase extractor
embedding_model = SentenceTransformer(
    'paraphrase-multilingual-MiniLM-L12-v2',
)

phrase_extractor = KeyBERT(model=embedding_model)

In [7]:
# Utility function to fix common character format errors
def clean(text):
    cleaned_text = html.unescape(text)
    substitutions = [
        (r' \. ', ''),
        (r'\xa0', ' '),
        (r'\.\s{2,}', '.\n',),
        (r'\n\s*[–-]\s*', '')
    ]
    for pattern, repl in substitutions:
        cleaned_text = re.sub(pattern, repl, cleaned_text)

    return cleaned_text

In [8]:
dataset = {}
max_file_count = 100
sentence_count = 100

for en_path in en_paths:
    # Cap the dataset size
    if len(dataset) >= max_file_count:
        break

    # Read in a file's English sentences
    with open(en_path, 'r') as en_file:
        en_sentences = clean(en_file.read()).strip().split('\n')

    # Read in the corresponding Spanish sentences
    es_path = str(en_path.replace('/en', '/es')).replace('-en', '-es')
    with open(es_path, 'r') as es_file:
        es_sentences = clean(es_file.read()).strip().split('\n')

    file_name = Path(en_path).name.replace('-en.txt', '')

    # As long as the sentence counts match between the English and Spanish files:
    if len(en_sentences) >= sentence_count:
        if len(en_sentences) != len(es_sentences):
            print(f'Note: {file_name} was skipped due to inconsistent sentence counts')
        else:
            # Create sentence pair entry and associate with the parent document

            en_sentences = en_sentences[:sentence_count]
            es_sentences = es_sentences[:sentence_count]

            document_entry = []

            # Insert sentences with calculated positional boundaries
            en_start = 0
            es_start = 0
            for i in range(sentence_count):
                en_sentence = en_sentences[i]
                es_sentence = es_sentences[i]

                en_end = en_start + len(en_sentence)
                es_end = es_start + len(es_sentence)

                sentence_entry = {
                    'en': en_sentence.strip(),
                    'es': es_sentence.strip(),
                    'en_bounds': (en_start, en_end),
                    'es_bounds': (es_start, es_end)
                }

                # Assume next sentence starts after 1 char gap
                en_start = en_end + 1
                es_start = es_end + 1

                document_entry.append(sentence_entry)

            dataset[file_name] = document_entry

Note: ep-05-06-09 was skipped due to inconsistent sentence counts
Note: ep-05-07-07 was skipped due to inconsistent sentence counts
Note: ep-05-01-12 was skipped due to inconsistent sentence counts
Note: ep-05-09-29 was skipped due to inconsistent sentence counts
Note: ep-06-03-16 was skipped due to inconsistent sentence counts
Note: ep-03-11-18 was skipped due to inconsistent sentence counts
Note: ep-03-12-03 was skipped due to inconsistent sentence counts
Note: ep-04-11-18 was skipped due to inconsistent sentence counts
Note: ep-03-12-15 was skipped due to inconsistent sentence counts
Note: ep-04-02-11 was skipped due to inconsistent sentence counts
Note: ep-06-06-01 was skipped due to inconsistent sentence counts
Note: ep-04-01-28 was skipped due to inconsistent sentence counts


In [9]:
# Example sentence pair
dataset[file_name][0]

{'en': 'Resumption of the session I declare resumed the session of the European Parliament adjourned on Friday, 12 February 1999.',
 'es': 'Reanudación del período de sesiones Declaro reanudado el periodo de sesiones del Parlamento Europeo, interrumpido el viernes 12 de febrero de 1999.',
 'en_bounds': (0, 121),
 'es_bounds': (0, 147)}

Note that the dataset still lacks key phrase data. To avoid the high startup costs associated with calling the KeyBERT extractor multiple times, we have compiled the dataset's sentences beforehand—allowing us to flatten the extractor's input and generate all key phrases in one pass.

In [10]:
# Flatten the dataset sentences into a single list
flat_sentences = []
for doc in dataset.values():
    for sent_pair in doc:
        flat_sentences.append(sent_pair['en'])
        flat_sentences.append(sent_pair['es'])

# Extract 3-gram keyphrases
keyphrases = phrase_extractor.extract_keywords(
    flat_sentences,
    keyphrase_ngram_range=(1, 3), # Limit keyphrases to 3 words
    stop_words=None,
    top_n=20
)

To keep the key phrases focused, we will filter out any key phrases that have a difference in similarity score of 0.05 or higher compared to the top phrase's similarity score.

In [11]:
# Filter out keyphrases with similarity scores 0.05+ lower than the top score
for i, phrase_list in enumerate(keyphrases):
    # phrase_list is sorted, so top phrase is always index 0
    max_score = phrase_list[0][1]
    score_cutoff = max_score - 0.05

    # Get just the key phrases within 0.05 of the top score
    keyphrases[i] = list(map(
        lambda w: w[0],
        filter(lambda kw: kw[1] >= score_cutoff, phrase_list)
    ))

Next, we associate the flattened key phrases with their parent documents and sentences within the dataset.

In [12]:
en_phrases = []
es_phrases = []

# Split key phrases by their languages
for i in range(len(keyphrases)):
    # English doc
    if i % 2 == 0:
        en_phrases.append(keyphrases[i])
    # Spanish doc
    else:
        es_phrases.append(keyphrases[i])

# Batch key phrases into their documents
en_phrase_batches = list(batched(en_phrases, sentence_count))
es_phrase_batches = list(batched(es_phrases, sentence_count))

# Insert the keyphrases into the dataset
for i, doc in enumerate(dataset.values()):
    # Each batch is a tuple of keyphrase lists
    curr_en_batch = en_phrase_batches[i]
    curr_es_batch = es_phrase_batches[i]

    for j, sent_object in enumerate(doc):
        # Each _phrases list corresponds to a sentence
        curr_en_phrases = curr_en_batch[j]
        curr_es_phrases = curr_es_batch[j]
        sent_object['en_phrases'] = curr_en_phrases
        sent_object['es_phrases'] = curr_es_phrases

Now we can export the dataset.

In [13]:
with open('data/sentence_dataset.pkl', 'wb') as file:
    pickle.dump(dataset, file)

with open('data/sentence_dataset.pkl', 'rb') as file:
    saved_dataset = pickle.load(file)

Example sentence pair:

In [14]:
saved_dataset[file_name][0]

{'en': 'Resumption of the session I declare resumed the session of the European Parliament adjourned on Friday, 12 February 1999.',
 'es': 'Reanudación del período de sesiones Declaro reanudado el periodo de sesiones del Parlamento Europeo, interrumpido el viernes 12 de febrero de 1999.',
 'en_bounds': (0, 121),
 'es_bounds': (0, 147),
 'en_phrases': ['session declare resumed',
  'european parliament adjourned',
  'adjourned',
  'parliament adjourned on',
  'parliament adjourned',
  'adjourned on'],
 'es_phrases': ['sesiones declaro reanudado', 'reanudación del período']}